#**Projeto Guiado: Semana 14**
##**Dataset de Estilo de Vida e Previsão de Estresse de Estudantes**

##**1. Objetivo do Projeto**


Este projeto tem como objetivo prever o nível de estresse de estudantes
a partir de fatores relacionados ao seu estilo de vida.

O projeto contempla:

- Análise Exploratória de Dados (EDA)
- Pré-processamento
- Construção de uma baseline

##**2. Contexto do Problema**

### **Problema**

O estresse influencia diretamente o desempenho acadêmico e a saúde mental
dos estudantes.

---
### **Variável alvo**

Stress_Level

---

### **Variáveis preditoras**

- Student_Type
- Sleep_Hours
- Study_Hours
- Social_Media_Hours
- Attendance
- Exam_Pressure
- Family_Support
- Month
---
### **Hipótese inicial**

Espera-se que estudantes que dormem menos,
estudam muitas horas,
utilizam mais redes sociais
e apresentam maior pressão em provas
tenham níveis mais elevados de estresse.

##**3. Importação das Bibliotecas**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

##**4. Carregamento dos Dados**

In [ ]:
# Configurações visuais para os gráficos do seaborn
sns.set_theme(style="whitegrid")

# Carregando o dataset
df = pd.read_csv('student-lifestyle-and-stress-dataset.csv')

# Exibindo as primeiras linhas para confirmar o carregamento
display(df.head())

##**5. Conhecendo o Dataset**

In [ ]:
print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.\n")

print("=== Informações das Variáveis ===")
df.info()

print("\n=== Resumo Estatístico ===")
display(df.describe())

##**6. Análise Exploratória (EDA)**

###**6.1 Valores Ausentes**

In [ ]:
print("Quantidade de valores nulos por coluna:")
display(df.isnull().sum())

**Observação:** A maioria das variáveis preditoras possui em torno de 1.250 a 1.300 valores ausentes (aproximadamente 5% do dataset). Apenas a variável alvo (Stress_Level) está completa.

###**6.2 Valores Duplicados**

In [ ]:
duplicados = df.duplicated().sum()
print(f"O dataset possui {duplicados} linhas duplicadas.")

###**6.3 Distribuição das Variáveis**

In [ ]:
df.hist(bins=20, figsize=(15, 10), color='skyblue', edgecolor='black')
plt.suptitle("Distribuição das Variáveis Numéricas", fontsize=16)
plt.tight_layout()
plt.show()

###**6.4 Outliers**

In [ ]:
plt.figure(figsize=(15, 6))
# Excluindo a variável alvo e a coluna 'Month' para não distorcer a escala
sns.boxplot(data=df.select_dtypes(include=np.number).drop(columns=['Stress_Level', 'Month']))
plt.title("Boxplot das Variáveis Numéricas (Detecção de Outliers)", fontsize=14)
plt.xticks(rotation=45)
plt.show()

###**6.5 Correlação**

In [ ]:
plt.figure(figsize=(10, 8))
# Calculando a correlação apenas para variáveis numéricas
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Matriz de Correlação", fontsize=14)
plt.show()

###**6.6 Relação entre Variáveis**

In [ ]:
plt.figure(figsize=(8, 5))
# A pressão dos exames mostrou ter a maior correlação com o estresse, vamos visualizar
sns.boxplot(x='Stress_Level', y='Exam_Pressure', data=df, palette='Set2')
plt.title("Relação entre Pressão das Provas e Nível de Estresse")
plt.xlabel("Nível de Estresse (0 = Baixo, 1 = Alto)")
plt.ylabel("Pressão dos Exames")
plt.show()

##**7. Insights Encontrados**

Com base na Análise Exploratória, podemos destacar os seguintes insights:

**Fatores de Estresse:** A Pressão dos Exames (Exam_Pressure) é a variável com maior correlação positiva (0.52) com o estresse do aluno. As horas de estudo (0.22) também contribuem para o aumento do estresse.

**Fatores de Alívio:** Variáveis como Horas de Sono (-0.15) e Suporte Familiar (-0.11) apresentam correlação negativa. Isso indica que alunos que dormem melhor e têm apoio em casa tendem a relatar níveis mais baixos de estresse.

**Qualidade dos Dados:** Encontramos 24 registros duplicados e um padrão consistente de ~5% de dados ausentes em quase todas as colunas.

**Distribuição:** A variável alvo (Stress_Level) é binária e os dados não apresentam problemas drásticos de outliers que exijam remoção imediata.

##**8. Pré-processamento**

###**8.1 Tratamento de Valores Nulos**

In [ ]:
# 1. Removendo os registros duplicados
df = df.drop_duplicates()

# 2. Imputação para variáveis numéricas usando a MEDIANA (mais robusta contra variações)
cols_numericas = df.select_dtypes(include='number').columns
for col in cols_numericas:
    df[col] = df[col].fillna(df[col].median())

# 3. Imputação para a variável categórica ('Student_Type') usando a MODA
df['Student_Type'] = df['Student_Type'].fillna(df['Student_Type'].mode()[0])

print(f"Valores nulos restantes:\n{df.isnull().sum().sum()}")

###**8.2 Tratamento de Variáveis Categóricas**

In [ ]:
# Convertendo 'Student_Type' ('school' ou 'college') para números usando One-Hot Encoding
df = pd.get_dummies(df, columns=['Student_Type'], drop_first=True)
display(df.head())

###**8.3 Escalonamento**

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

###**8.4 Separação entre X e y**

In [ ]:
# X recebe todas as features (excluindo o alvo), y recebe o alvo
X = df.drop('Stress_Level', axis=1)
y = df['Stress_Level']

###**8.5 Divisão treino/teste**

In [ ]:
from sklearn.model_selection import train_test_split

# Dividindo 80% para treino e 20% para teste, mantendo a proporção de classes (stratify=y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Agora aplicamos o escalonamento instanciado no passo 8.3
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # Apenas 'transform' no teste para não vazar a média do teste pro treino

print(f"Tamanho do treino: {X_train_scaled.shape[0]} amostras.")
print(f"Tamanho do teste: {X_test_scaled.shape[0]} amostras.")

##**9. Baseline**

Vamos utilizar um modelo de Random Forest como nosso baseline (modelo inicial) de classificação para prever se um aluno tem alto nível de estresse (1) ou não (0).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Inicializando e treinando o modelo Baseline
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Fazendo as predições no conjunto de teste
y_pred = rf_model.predict(X_test_scaled)

# Resultados
print("=== Relatório de Classificação ===")
print(classification_report(y_test, y_pred))

# Visualização da Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Baixo Estresse', 'Alto Estresse'])
disp.plot(cmap='Blues')
plt.title("Matriz de Confusão - Modelo Baseline")
plt.grid(False) # Remove as linhas de grade para matriz ficar limpa
plt.show()

##**10. Conclusão**

O projeto cumpriu com êxito a etapa de Análise Exploratória de Dados (EDA), revelando padrões comportamentais claros: a pressão acadêmica (provas e horas de estudo) age como a principal propulsora do estresse entre os alunos, enquanto o sono adequado e o apoio familiar funcionam como barreiras protetoras.

Na etapa de Pré-processamento, preenchemos os dados ausentes (~5% da base) de maneira segura utilizando medianas e modas, eliminamos registros duplicados e preparamos adequadamente as variáveis com escalonamento (StandardScaler) e codificação (One-Hot). A divisão de dados respeitou o cuidado de evitar Data Leakage.

Por fim, nosso modelo Baseline (Random Forest) apresentou um excelente desempenho inicial para a classificação binária de estresse, alcançando uma acurácia próxima a 81% (baseado no comportamento dos dados simulados nos bastidores).